# M20b — UCI Appliances Energy Prediction

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Apply the same four structural paths to the Appliances Energy time series.

**Provenance.** The target, horizon, dimensions, reservoir size, candidate count, split lengths, and ridge coefficient are recovered from the manuscript. The original notebook binary is unavailable. This notebook reconstructs the transparent preprocessing consistent with the reported `19729` rows and `32` input features. It executes automatically when `data/energydata_complete.csv` is present.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
DATA=ROOT/"data"/"energydata_complete.csv"
URL="https://archive.ics.uci.edu/static/public/374/appliances%2Benergy%2Bprediction.zip"
HORIZON=6; N=50; K=13; RIDGE=1e-4; SEED=20260718
hist=pd.read_csv(FROZEN/"real_world_historical_reference.csv")
display(hist[hist.dataset=="Appliances Energy"])
print("data file:",DATA)
print("present:",DATA.exists())

,dataset,prepared_rows,features,target,horizon,temperature_support_range,temperature_safe_gain,temperature_safe_minus_shuffled,ci_low,ci_high,safe_dispersion_corr,anchor_spread_corr
0,Appliances Energy,19729,32,log(1+future Appliances),h=6 samples (~1 hour),45.62,0.00341,-0.01199,-0.05331,0.01532,0.51,0.522


data file: /mnt/data/temperature-calibrated-reservoirs-reproducibility/data/energydata_complete.csv
present: False


## Preprocessing

In [3]:
def prepare_appliances(path):
    df=pd.read_csv(path)
    dt=pd.to_datetime(df["date"])
    numeric=df.drop(columns=["date"]).apply(pd.to_numeric,errors="coerce")
    # All 28 contemporaneous numeric measurements are retained, including current Appliances.
    X=numeric.copy()
    X["hour_sin"]=np.sin(2*np.pi*dt.dt.hour/24)
    X["hour_cos"]=np.cos(2*np.pi*dt.dt.hour/24)
    X["dow_sin"]=np.sin(2*np.pi*dt.dt.dayofweek/7)
    X["dow_cos"]=np.cos(2*np.pi*dt.dt.dayofweek/7)
    y=np.log1p(numeric["Appliances"].shift(-HORIZON))
    X=X.iloc[:-HORIZON].reset_index(drop=True); y=y.iloc[:-HORIZON].reset_index(drop=True)
    assert len(X)==19729 and X.shape[1]==32, (len(X),X.shape)
    return X.to_numpy(float),y.to_numpy(float)

## Execute when the public data file is available

In [4]:
if not DATA.exists():
    print("SKIPPED in this execution environment: public dataset is not mounted.")
    print("Place energydata_complete.csv in data/ and Restart & Run All.")
else:
    X,y=prepare_appliances(DATA)
    # Use the first reported-length chronological block; scaling is learned only from training.
    ntr,nv,nt=2500,1000,1000
    Xtr,Xv,Xt=X[:ntr],X[ntr:ntr+nv],X[ntr+nv:ntr+nv+nt]
    ytr,yv,yt=y[:ntr],y[ntr:ntr+nv],y[ntr+nv:ntr+nv+nt]
    mu,sd=Xtr.mean(0),Xtr.std(0)+1e-12
    Xtr=(Xtr-mu)/sd; Xv=(Xv-mu)/sd; Xt=(Xt-mu)/sd
    rows=[]
    for path in ["temperature","gain","leak","sparsity"]:
        rows.append(tcr.evaluate_arrays(Xtr,ytr,Xv,yv,Xt,yt,"appliances",0,path,SEED,N,K,RIDGE))
    rep=pd.DataFrame(rows); rep.to_csv(REPRO/"m20b_replication_summary.csv",index=False)
    display(rep.round(6))

SKIPPED in this execution environment: public dataset is not mounted.
Place energydata_complete.csv in data/ and Restart & Run All.


Official source: UCI Appliances Energy Prediction. The repository does not redistribute the dataset; see `data/README.md`.